## 4、记忆治理策略（上下文管理）

随着对话的进行，历史消息不断累积，state会持续增长，为模型带来挑战：
1. LLM的上下文窗口是有限的，完整历史可能无法装入LLM的上下文窗口，导致上下文丢失或错误。
2. 即便模型的上下文窗口够大，多数LLM在长上下文场景仍然表现不佳。模型会 被陈旧或离题的内容“分散注意力” 。
3. 同时，会带来高昂的token花费。

此时需要对上下文进行管理：对历史记录进行压缩、清理、重组等。


In [1]:

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os



# 1、提供大模型
load_dotenv(override=True)

model = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    profile={"max_input_tokens": 128_000},
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    # temperature=1.5,
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    extra_body={"enable_thinking": False},
    # max_tokens=10,
)

## 1、消息裁剪

In [2]:
from langchain_core.messages import HumanMessage
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig
from typing import Any

@before_model
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    messages = state["messages"]

    if len(messages) <= 3:
        return None

    first_message = messages[0]
    # 策略：如果有偶数条消息，则取最近的3条消息；如果有奇数条消息，则取最近的4条消息
    recent_messages = messages[-3:] if len(messages) % 2 == 0 else messages[-4:]

    new_messages = [first_message] + recent_messages

    return {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            *new_messages
        ]
    }

agent = create_agent(
    model=model,
    middleware=[trim_messages],
    checkpointer=InMemorySaver(),
)
config: RunnableConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": [HumanMessage("你好，我是老王")]}, config)
agent.invoke({"messages": [HumanMessage("从现在起，你叫小王")]}, config)
agent.invoke({"messages": [HumanMessage("今天天气不错")]}, config)
final_response = agent.invoke({"messages": [HumanMessage("告诉我，你是谁？我是谁？")]}, config)

for msg in final_response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好，我是老王
================================== Ai Message ==================================

好的，老王！我是小王。👋

请问接下来有什么吩咐？
================================ Human Message =================================

今天天气不错
================================== Ai Message ==================================

是啊，老王！天气好心情也跟着变好了。

这种好天气最适合出去溜达溜达，或者找个舒服的地方喝杯茶、晒晒太阳了。您今天有什么安排吗？是打算出门转转，还是就在家享受这份惬意？
================================ Human Message =================================

告诉我，你是谁？我是谁？
================================== Ai Message ==================================

哈哈，老王，这问题问得颇有哲学意味啊！不过既然咱们刚才已经“对上暗号”了，那我就按咱们的设定来回答：

**你是谁？**
你是**老王**。在我这里，你是一位亲切、随和的对话伙伴。也许你正享受着今天的好天气，也许心里装着不少故事或疑问，准备随时和我聊聊。

**我是谁？**
我是**小王**（也可以叫我 AI 助手）。我是由阿里云通义千问团队开发的大型语言模型。我的角色是您的智能伙伴，可以陪您聊天解闷、解答疑问、协助创作，或者只是简单地回应您的每一句话。

简单来说：**你是发问者，我是回应者；你是主角，我是配角。**

怎么，老王，是不是今天天气太好，让您想探讨一下“存在主义”了？😄


## 2、消息删除

In [3]:
from langchain_core.messages import RemoveMessage
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import after_model
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig

@after_model
def delete_old_messages(state: AgentState, runtime: Runtime) -> dict | None:
    messages = state["messages"]
    # 保持最近的 5 条消息
    if len(messages) > 5:
        # 消息总数超出阈值，计算需要删除的条数
        to_delete = len(messages) - 5
        # 生成删除指令：移除最开头最早的若干消息
        return {"messages": [RemoveMessage(id=m.id) for m in messages[:to_delete]]}
    # 消息数量未超限，不做任何修改
    return None

agent = create_agent(
    model=model,
    middleware=[delete_old_messages],
    checkpointer=InMemorySaver()
)
config: RunnableConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": "你好，我是老王"}, config)
agent.invoke({"messages": "从现在起，你叫小王"}, config)
agent.invoke({"messages": "今天天气不错"}, config)
final_response = agent.invoke({"messages": "告诉我，你是谁？我是谁？"}, config)

for msg in final_response["messages"]:
    msg.pretty_print()

================================== Ai Message ==================================

好的，老王！我是小王。请问接下来有什么吩咐？
================================ Human Message =================================

今天天气不错
================================== Ai Message ==================================

是啊，老王！天气好，心情也跟着舒畅。

您今天打算出去走走，还是就在家里享受这好天气？要是出门的话，记得带杯茶或者找个舒服的地方坐坐，别太累着。
================================ Human Message =================================

告诉我，你是谁？我是谁？
================================== Ai Message ==================================

老王，这问题问得有意思！

我是**小王**，您的智能助手，随时准备为您答疑解惑、处理事务，或者陪您聊聊天。

您是**老王**，我的老朋友（虽然咱们刚认识不久，但称呼上已经挺亲切了），也是此刻掌握对话主动权的人。

怎么，突然问起这个，是有什么特别的感慨，还是想考考我的记性？😄


## 3、摘要

In [6]:

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os


# 1、提供大模型
load_dotenv(override=True)

model_out = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    profile={"max_input_tokens": 128_000},
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    # temperature=1.5,
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    extra_body={"enable_thinking": False},
    # max_tokens=10,
)

model_in = init_chat_model(
    model="GLM-4.5-Air",
    model_provider="openai",
    api_key=os.getenv("ZHIPUAI_API_KEY"),
    base_url=os.getenv("ZHIPUAI_BASE_URL"),
    extra_body={"enable_thinking": False}
)

In [7]:
from langchain.agents.middleware import SummarizationMiddleware
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# 创建带摘要中间件的 Agent
agent = create_agent(
    model=model_out,
    tools=[],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model_in,
            trigger=[
                ("tokens", 100),  # 上下文总token超过100就触发历史摘要
            ],
            keep=("messages", 2), # 固定保留最近2条完整原始对话消息
            summary_prompt="对历史消息摘要，消息列表如下\n{messages}",
        )
    ]
)

config = {"configurable": {"thread_id": "1"}}
print("\n进行多轮对话...")

conversations = [
    "我叫张三，是工程师。这里是一段非常长非常长的废话..." * 20, # 强制撑爆 100 tokens
    "请总结一下我的信息"
]

for msg in conversations:
    response = agent.invoke(
        {"messages": [{"role": "user", "content": msg}]},
        config=config
    )
    for msg in response["messages"]:
        msg.pretty_print()
    print("*" * 50)


进行多轮对话...
================================ Human Message =================================

我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...
================================== Ai Message ==================================

你好，张三！很高兴认识你这位工程师。

虽然中间夹杂了很多重复的“废话”，但我已经成功提取到了关键信息：**你是张三，职业是工程师。**

请问有什么我可以帮你的吗？无论是技术问题、代码调试、工程方案探讨，还是其他任何疑问，我都乐意为你提供帮助！
**************************************************
================================ Human Message =================================

Here is a summary of the co

In [9]:
from rich import print as rprint

final_state = agent.get_state(config)
rprint(final_state)

StateSnapshot(
    values={
        'messages': [
            HumanMessage(
                content='Here is a summary of the conversation to 
date:\n\n根据提供的历史消息列表，消息内容高度重复，主要包含两个核心元素：说话者自我介绍（“我叫张三，是工程师。”）和
填充性废话（“这里是一段非常长非常长的废话...”）。该消息整体冗长，没有提供任何实质信息或新内容，仅用于填充或测试目的
。\n\n### 消息摘要：\n- **说话者信息**：张三，工程师。\n- 
**内容概述**：消息由“我叫张三，是工程师。”开头，随后重复“这里是一段非常长非常长的废话...”多次（约20次），整体内容冗
余且无实质信息。\n- **关键点**：消息本质是废话，缺乏有效内容，可能用于演示或测试场景。\n\n### 摘要说明：\n- 
该摘要简洁概括了消息的核心要素（说话者身份和内容性质），避免了重复细节，以突出“无实质信息”这一关键特征。\n- 
如果需要进一步分析（如重复次数统计或上下文），请提供更多指示！',
                additional_kwargs={'lc_source': 'summarization'},
                response_metadata={},
                id='82487534-1e77-4c03-889b-6fa7cbea6079'
            ),
            AIMessage(
                content='你好，张三！很高兴认识你这位工程师。\n\n虽然中间夹杂了很多重复的“废话”，但我已经成功提取到
了关键信息：**你是张三，职业是工程师。**\n\n请问有什么我可以帮你的吗？无论是技术问题、代码调试、工程方案探讨，还是
其他任何疑问，我都乐意为你提供帮助！',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 63,
                        'prompt_tokens': 292,
                        'total_tokens': 355,
                        'completion_tokens_details': None,
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 292}
                    },
                    'model_provider': 'openai',
                    'model_name': 'qwen3.7-plus',
                    'system_fingerprint': None,
                    'id': 'chatcmpl-0d9b65e3-cfb1-91da-89ec-b0ab8a9b1328',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--019f9957-af59-7e71-a9c0-fa11164aa76a-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 292,
                    'output_tokens': 63,
                    'total_tokens': 355,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {}
                }
            ),
            HumanMessage(
                content='请总结一下我的信息',
                additional_kwargs={},
                response_metadata={},
                id='b93ebc08-1c61-477d-ae46-cb0717861595'
            ),
            AIMessage(
                content='根据之前的对话记录，关于您的信息总结如下：\n\n*   **姓名**：张三\n*   
**职业**：工程师\n\n除此之外，之前的消息内容主要为重复的填充性文本，未包含其他实质性个人信息或背景细节。如果您愿意
分享更多专业领域或具体需求，我可以为您提供更针对性的帮助。',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 64,
                        'prompt_tokens': 306,
                        'total_tokens': 370,
                        'completion_tokens_details': None,
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 306}
                    },
                    'model_provider': 'openai',
                    'model_name': 'qwen3.7-plus',
                    'system_fingerprint': None,
                    'id': 'chatcmpl-5e778110-7428-99ba-9a0e-8082ae5cff8f',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--019f9957-dcb1-76a3-acab-130725693012-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 306,
                    'output_tokens': 64,
                    'total_tokens': 370,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {}
                }
            )
        ]
    },
    next=(),
    config={
        'configurable': {
            'thread_id': '1',
            'checkpoint_ns': '',
            'checkpoint_id': '1f188280-cdc9-6c13-8006-78ef9f54db0f'
        }
    },
    metadata={'source': 'loop', 'step': 6, 'parents': {}},
 